# Training a CE model

using the provided dataset. Script to generate said dataset with a MACE foundation model is included. Keep in mind that the accuracy of said foundation model is not necessarily given for any systems.

Participants should get a feeling for:
 - the hyperparameters (cutoffs, which interactions to include, much more important as for MACE)
 - what accuracy to expect (will not be nearly as good as MACE)

## Setting up the Cluster Space

A cluster expansion model requires the specification of a primitive cell defining the crystal lattice. All the structures in the data set must align to this lattice. The reference data itself is obtained via

In [ ]:

from icet import ClusterSpace, StructureContainer, ClusterExpansion
from trainstation import CrossValidationEstimator
from pathlib import Path
import ase.io
from icet.tools import map_structure_to_reference
from tqdm import tqdm


# This sets the sublattices in the CE model, multiple lists can be given for different sublattices at different sites
chemical_symbols = ["Cu", "Ni"]

# this specifies which dataset we are using by providing the maximum number of atoms to train on
max_atom_num = 8

base_path = Path.cwd().parents[1]
struct_path = (
    base_path
    / "data"
    / f"CE_dataset_{chemical_symbols[0]}{chemical_symbols[1]}"
)
dataset_path = (
    struct_path 
    / f"enumerated_structures_{max_atom_num}_calculated.extxyz"
)


# load the structures in the dataset
dataset = ase.io.read(dataset_path, index=":")


# load the relaxed primitive structure from which we generated the dataset
prim = dataset[0]


# we create a cluster space with certain cutoffs for pairs, triplets and quadruplets
cs = ClusterSpace(
    structure=prim,
    cutoffs=[13, 7, 6],
    chemical_symbols=[chemical_symbols],
)

# the structure container needs to know the basic conditions of the cluster space
# the dataset will then be added based on the basic lattice
sc = StructureContainer(cluster_space=cs)


for aid, atoms in tqdm(enumerate(dataset)):
    # Note: the structures need positions and cell parameters to align to the primitive structure
    # we can use map_structure_to_reference to map the structure to the primitive
    # in our case, the positions are already on-lattice
    # atoms, info = map_structure_to_reference(atoms, prim)
    mapped = atoms
    sc.add_structure(
        atoms, properties=dict(mixing_energy=atoms.info["mixing_energy"])
    )


## Training the model

Next, we will train the model, this should only take a few seconds. One major parameter to choose is the regularization parameter $\lambda$ it will influence the sparsity of the solution. Check the notebook 2.1 for a code to visualize the behavior. Here, it can make sense to choose an improved setting for your system.

In [ ]:
import numpy as np

seed = np.random.randint(0, 1e5)
print("seed:", seed)
A, y = sc.get_fit_data(key="mixing_energy")
eopt = CrossValidationEstimator(
    (A, y),
    fit_method="ardr", # the algorithm to use
    seed=seed,
    threshold_lambda=1e4,
)
eopt.validate()
eopt.train()
print(eopt)
ce = ClusterExpansion(
    cluster_space=cs, parameters=eopt.parameters, metadata=eopt.summary
)
print(ce)
model_output_path = struct_path / f"ce_model_{max_atom_num}.ce"
ce.write(model_output_path)

## Using the model and evaluating accuracy metrics

Similar to training MACE, we can create some parity plots to assess the models performance.

In [ ]:
# here, we will create a parity plot to judge the model's performance

import matplotlib.pyplot as plt
ref_energies = []
model_energies = []
species_fractions = []
for atoms in tqdm(dataset):
    y_pred = ce.predict(atoms)
    ref_energies.append(atoms.info["mixing_energy"])
    model_energies.append(y_pred)
    species_fractions.append(np.sum(atoms.symbols == chemical_symbols[1]) / len(atoms))

ref_energies = np.asarray(ref_energies)
model_energies = np.asarray(model_energies)
RMSE = np.sqrt(np.mean((ref_energies - model_energies) ** 2))
print(f"RMSE: {RMSE}")
MAE = np.mean(np.abs(ref_energies - model_energies))
print(f"MAE: {MAE}")

plt.scatter(ref_energies, model_energies)
plt.axline([np.min(ref_energies), np.min(ref_energies)], slope=1, color="k")
plt.xlabel("Reference / eV/atom")
plt.ylabel("Prediction / eV/atom")
plt.show()

Clearly, the same level of agreement as we saw for MACE models is more difficult to achieve. The situation is generally worse for elements with vastly differing volumes. We can then also compute the convex hull for the dataset. Does the small dataset for your system show a significant miscibility gap?

In [ ]:
from icet.tools import ConvexHull


plt.plot(species_fractions, ref_energies, 'o', color='blue', label="Ref")
hull_ref = ConvexHull(species_fractions, ref_energies)
plt.plot(hull_ref.concentrations, hull_ref.energies, '-o', color='orange', label='Ref hull')

plt.plot(species_fractions, model_energies, 'x', color='green', label='CE')
hull_model = ConvexHull(species_fractions, model_energies)
plt.plot(hull_model.concentrations, hull_model.energies, '-x', color='red', label='CE hull')

plt.xlabel(f"{chemical_symbols[1]} fraction")
plt.ylabel("Energy of mixing / eV/atom")
plt.legend()
plt.show()
